In [1]:
import math, time, os, csv, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import BertTokenizerFast


class Clos(nn.Module):
    def __init__(self, in_features=768, out_features=None, channel=3,
                 switches=None, bias=True, middle_switch_multiplier=4):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features or in_features
        self.channel      = channel
        self.use_bias     = bias
        self.msm          = middle_switch_multiplier
        self.sw           = {}
        self._find_factors()
        if switches:
            self.sw.update(switches)
        self.w1 = nn.Parameter(torch.Tensor(self.sw["bin"], self.sw["b1"], self.sw["b2"]))
        self.w2 = nn.Parameter(torch.Tensor(self.sw["b1"],  self.sw["b2"], self.sw["b3"]))
        self.w3 = nn.Parameter(torch.Tensor(self.sw["b2"],  self.sw["b3"], self.sw["bout"]))
        if bias:
            self.b1 = nn.Parameter(torch.Tensor(self.sw["b1"]))
            self.b2 = nn.Parameter(torch.Tensor(self.sw["b2"]))
            self.b3 = nn.Parameter(torch.Tensor(self.sw["b3"]))
        else:
            self.register_parameter("b1", None)
            self.register_parameter("b2", None)
            self.register_parameter("b3", None)
        self._reset()

    def _find_factors(self):
        for i in range(int(math.sqrt(self.in_features)), 0, -1):
            if self.in_features % i == 0:
                self.sw["bin"] = i
                self.sw["b1"]  = self.in_features // i
                break
        for i in range(int(math.sqrt(self.out_features)), 0, -1):
            if self.out_features % i == 0:
                self.sw["bout"] = i
                self.sw["b3"]   = self.out_features // i
                break
        self.sw["b2"] = self.msm * self.sw["bin"]

    def _reset(self):
        for w in [self.w1, self.w2, self.w3]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        if self.use_bias:
            for w, b in [(self.w1, self.b1), (self.w2, self.b2), (self.w3, self.b3)]:
                fan_in, _ = nn.init._calculate_fan_in_and_fan_out(w)
                bd = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
                nn.init.uniform_(b, -bd, bd)

    def _fwd3(self, x):
        b, c, _ = x.shape
        x = x.view(b, c, self.sw["bin"], self.sw["b1"])
        if self.use_bias:
            x = torch.einsum("bcnr,nrm->bcmr", x, self.w1) + self.b1
            x = torch.einsum("bcmr,rmn->bcnm", x, self.w2) + self.b2
            x = torch.einsum("bcnm,mro->bcor", x, self.w3) + self.b3
        else:
            x = torch.einsum("bcnr,nrm->bcmr", x, self.w1)
            x = torch.einsum("bcmr,rmn->bcnm", x, self.w2)
            x = torch.einsum("bcnm,mro->bcor", x, self.w3)
        return x.reshape(b, c, -1)

    def _fwd2(self, x):
        b = x.shape[0]
        x = x.view(b, self.sw["bin"], self.sw["b1"])
        if self.use_bias:
            x = torch.einsum("bnr,nrm->bmr", x, self.w1) + self.b1
            x = torch.einsum("bmr,rmn->bnm", x, self.w2) + self.b2
            x = torch.einsum("bnm,mro->bor", x, self.w3) + self.b3
        else:
            x = torch.einsum("bnr,nrm->bmr", x, self.w1)
            x = torch.einsum("bmr,rmn->bnm", x, self.w2)
            x = torch.einsum("bnm,mro->bor", x, self.w3)
        return x.reshape(b, -1)

    def forward(self, x):
        return self._fwd3(x) if self.channel == 3 else self._fwd2(x)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def distill_to_clos(fc, channel=3, max_steps=3000, lr=2e-3, msm=4,
                    probe_rand=8192, probe_batch=512,
                    gate_margin=0.20, lam_eye=5.0, lam_gate=1.0,
                    seed=None, log_every=50):
    device = next(fc.parameters()).device
    dtype  = next(fc.parameters()).dtype
    if seed is not None:
        torch.manual_seed(seed)
    in_f  = fc.in_features
    out_f = fc.out_features
    clos  = Clos(in_f, out_f, channel=channel, bias=True,
                 middle_switch_multiplier=msm).to(device=device, dtype=dtype)
    fc.eval()
    clos.train()
    with torch.no_grad():
        eye   = torch.eye(in_f, device=device, dtype=dtype)
        zero  = torch.zeros(256, in_f, device=device, dtype=dtype)
        rand  = torch.randn(probe_rand, in_f, device=device, dtype=dtype)
        X     = torch.cat([zero, eye, -eye, rand], dim=0)
        T     = fc(X)
        T_eye = fc(eye)
    n_struct = 256 + 2 * in_f

    def fwd(x):
        return clos(x.unsqueeze(1)).squeeze(1) if channel == 3 else clos(x)

    opt   = AdamW(clos.parameters(), lr=lr, weight_decay=1e-4)
    sched = CosineAnnealingLR(opt, T_max=max_steps)
    print("    Step      TotalLoss    MSE          Gate         EyeMSE       Viol%    LR")
    print("    " + "-" * 80)
    for step in range(max_steps):
        k_s = min(probe_batch // 2, n_struct)
        k_r = probe_batch - k_s
        idx = torch.cat([
            torch.randint(0, n_struct,         (k_s,), device=device),
            torch.randint(n_struct, X.size(0), (k_r,), device=device),
        ])
        xb, tb = X[idx], T[idx]
        opt.zero_grad(set_to_none=True)
        sb        = fwd(xb)
        loss_mse  = F.mse_loss(sb, tb)
        sign      = torch.sign(tb)
        sign[sign == 0] = 1
        loss_gate = F.relu(gate_margin - sign * sb).mean()
        loss_eye  = F.mse_loss(fwd(eye), T_eye)
        loss      = loss_mse + lam_gate * loss_gate + lam_eye * loss_eye
        loss.backward()
        torch.nn.utils.clip_grad_norm_(clos.parameters(), 1.0)
        opt.step()
        sched.step()
        if step % log_every == 0 or step == max_steps - 1:
            with torch.no_grad():
                eye_mse  = F.mse_loss(fwd(eye), T_eye).item()
                viol_pct = (F.relu(gate_margin - sign * sb) > 0).float().mean().item() * 100
                cur_lr   = opt.param_groups[0]["lr"]
            print(f"    {step:5d}  {loss.item():12.4e}  {loss_mse.item():12.4e}  "
                  f"{loss_gate.item():12.4e}  {eye_mse:12.4e}  {viol_pct:5.1f}%  {cur_lr:.1e}")
    clos.eval()
    return clos


def replace_layers(model, layer_ids, replace_q=False, replace_k=False,
                   replace_v=False, replace_ffn=False,
                   distill_steps=3000, log_every=50):
    device = next(model.parameters()).device

    def swap(parent, attr, label):
        fc     = getattr(parent, attr)
        before = sum(p.numel() for p in fc.parameters())
        print(f"\n  Distilling {label}  [{fc.in_features} -> {fc.out_features}]  ({before:,} params)")
        clos = distill_to_clos(fc.to(device), channel=3,
                               max_steps=distill_steps, log_every=log_every)
        setattr(parent, attr, clos)
        after = clos.n_params()
        print(f"  Done: {after:,} params  (saved {before - after:,} = {100*(before-after)/before:.1f}%)")

    for lid in layer_ids:
        enc  = model.bert.encoder.layer[lid]
        attn = enc.attention.self
        print(f"\n  Layer {lid}  " + "-" * 40)
        if replace_q:
            swap(attn, "query", f"L{lid}.Q")
        if replace_k:
            swap(attn, "key",   f"L{lid}.K")
        if replace_v:
            swap(attn, "value", f"L{lid}.V")
        if replace_ffn:
            swap(enc.intermediate, "dense", f"L{lid}.FFN-inter")
            swap(enc.output,       "dense", f"L{lid}.FFN-out")
    return model


class MLMDataset(Dataset):
    def __init__(self, n_samples=300, max_length=128, mask_prob=0.15):
        from datasets import load_dataset
        tok    = BertTokenizerFast.from_pretrained("bert-base-uncased")
        raw    = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
        texts  = [t for t in raw["text"] if len(t.strip()) > 20][: n_samples * 5]
        enc    = tok(texts, max_length=max_length, padding="max_length",
                     truncation=True, return_tensors="pt")
        ids    = enc["input_ids"]
        attn   = enc["attention_mask"]
        labels = ids.clone()
        rand   = torch.rand(ids.shape)
        special = ((ids == tok.cls_token_id) | (ids == tok.sep_token_id) |
                   (ids == tok.pad_token_id))
        mi = (rand < mask_prob) & ~special
        ids[mi & (rand < mask_prob * 0.8)] = tok.mask_token_id
        rflag  = mi & (rand >= mask_prob * 0.8) & (rand < mask_prob * 0.9)
        ids[rflag] = torch.randint(1000, tok.vocab_size, (rflag.sum().item(),))
        labels[~mi] = -100
        has = mi.any(dim=1)
        self.ids    = ids[has][:n_samples]
        self.attn   = attn[has][:n_samples]
        self.labels = labels[has][:n_samples]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return {
            "input_ids":      self.ids[i],
            "attention_mask": self.attn[i],
            "labels":         self.labels[i],
        }


@torch.no_grad()
def eval_mlm(model, dataset, batch_size=16, device=None, n_timing=20, n_preview=5):
    if device is None:
        device = next(model.parameters()).device
    model.eval().to(device)
    tok        = BertTokenizerFast.from_pretrained("bert-base-uncased")
    loader     = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    total_loss = 0.0
    correct    = 0
    total      = 0
    shown      = 0
    print("\n  " + "-" * 60)
    print(f"  predict example (first {n_preview}):")
    print("  " + "-" * 60)
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        lbl  = batch["labels"].to(device)
        out  = model(input_ids=ids, attention_mask=attn, labels=lbl)
        mi   = lbl != -100
        pred = out.logits.argmax(-1)
        correct    += (pred[mi] == lbl[mi]).sum().item()
        total      += mi.sum().item()
        total_loss += out.loss.item() * mi.sum().item()
        if shown < n_preview:
            for b in range(ids.size(0)):
                if shown >= n_preview:
                    break
                mpos = (lbl[b] != -100).nonzero(as_tuple=True)[0]
                if len(mpos) == 0:
                    continue
                sl   = int(attn[b].sum().item())
                toks = tok.convert_ids_to_tokens(ids[b, :sl].tolist())
                sent = tok.convert_tokens_to_string(toks)
                print(f"\n  [{shown + 1}] {sent[:120]}")
                for p in mpos[:4]:
                    p  = int(p.item())
                    ti = int(lbl[b, p].item())
                    pi = int(pred[b, p].item())
                    tt = tok.convert_ids_to_tokens([ti])[0]
                    pt = tok.convert_ids_to_tokens([pi])[0]
                    t3 = " / ".join(tok.convert_ids_to_tokens(
                             out.logits[b, p].topk(3).indices.tolist()))
                    ok = "true" if ti == pi else "false"
                    print(f"       pos {p:3d}: True=[{tt:<12}]  Predict=[{pt:<12}]  {ok}  top3=[{t3}]")
                shown += 1

    t0 = time.perf_counter()
    for i, b in enumerate(DataLoader(dataset, batch_size=batch_size)):
        if i >= n_timing:
            break
        model(input_ids=b["input_ids"].to(device),
              attention_mask=b["attention_mask"].to(device))
    lat = 1000 * (time.perf_counter() - t0) / min(n_timing, len(dataset) // batch_size)
    return {
        "accuracy":   round(100 * correct / total, 3),
        "loss":       round(total_loss / total, 4),
        "latency_ms": round(lat, 1),
        "params":     sum(p.numel() for p in model.parameters()),
    }


RESULTS_FILE = "results.csv"


def save_result(row):
    exists = os.path.isfile(RESULTS_FILE)
    with open(RESULTS_FILE, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if not exists:
            w.writeheader()
        w.writerow(row)


def print_result(row):
    print(f"  ID           : {row['id']}")
    print(f"  MLM Accuracy : {row['accuracy_%']:.3f}%")
    print(f"  MLM Loss     : {row['loss']:.4f}")
    print(f"  Params       : {row['params']:,}  ({row['reduction_%']:.2f}% бууралт)")
    print(f"  Latency      : {row['latency_ms']:.1f} ms/batch")
    print("-" * 55)


def load_base_params():
    if not os.path.isfile("base_params.json"):
        raise FileNotFoundError(
            "base_params.json олдсонгүй.\n"
            "Эхлээд  python T0_baseline.py  ажиллуулна уу.")
    with open("base_params.json") as f:
        return json.load(f)


def make_result_row(exp_id, desc, replace_q, replace_k, replace_v,
                    replace_ffn, layers, metrics, base_params):
    return {
        "id":          exp_id,
        "description": desc,
        "replace_q":   replace_q,
        "replace_k":   replace_k,
        "replace_v":   replace_v,
        "replace_ffn": replace_ffn,
        "layers":      str(layers),
        "params":      metrics["params"],
        "reduction_%": round(100 * (base_params - metrics["params"]) / base_params, 2),
        "accuracy_%":  metrics["accuracy"],
        "loss":        metrics["loss"],
        "latency_ms":  metrics["latency_ms"],
    }

In [2]:
import json, torch, sys
from torch.utils.data import DataLoader
from transformers import BertForMaskedLM, BertTokenizerFast
import time, os, csv

# Kaggle dataset path add
sys.path.append("/kaggle/input/datasets/nominnna/utils-py")

from utils import MLMDataset, eval_mlm, save_result, print_result


DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SAMPLES  = 300
BATCH_SIZE = 16


def main():
    print("  T0 - Baseline ")
    print(f"  Device: {DEVICE}")

    model = BertForMaskedLM.from_pretrained("bert-base-uncased")
    model.eval()

    base_params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {base_params:,}")

    dataset = MLMDataset(n_samples=N_SAMPLES)
    print(f"  {len(dataset)} sample ")

    metrics = eval_mlm(model, dataset, batch_size=BATCH_SIZE, device=DEVICE)

    result = {
        "id":          "T0",
        "description": "Baseline",
        "replace_q":   False,
        "replace_k":   False,
        "replace_v":   False,
        "replace_ffn": False,
        "layers":      "none",
        "params":      metrics["params"],
        "reduction_%": 0.0,
        "accuracy_%":  metrics["accuracy"],
        "loss":        metrics["loss"],
        "latency_ms":  metrics["latency_ms"],
    }

    print_result(result)
    save_result(result)

    with open("base_params.json", "w") as f:
        json.dump({
            "base_params":        base_params,
            "baseline_accuracy":  metrics["accuracy"],
            "baseline_loss":      metrics["loss"],
        }, f, indent=2)

    print("\n  base_params.json saved")


if __name__ == "__main__":
    main()

  T0 - Baseline 
  Device: cuda


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

  300 sample 
  predict example (first 5):

  [1] [CLS] homarplaced [MASK]rus, known as the [MASK] lobster or common lobster, is a species [MASK] clawed lobster from the 
       pos   3: True=[##us        ]  Predict=[##us        ]  true  top3=[##us / ##ius / ##cus]
       pos   4: True=[gamma       ]  Predict=[hume        ]  false  top3=[hume / ne / ho]
       pos  10: True=[european    ]  Predict=[red         ]  false  top3=[red / black / white]
       pos  19: True=[of          ]  Predict=[of          ]  true  top3=[of / a / or]

  [2] [CLS] ho [MASK]us gammarus is a large [MASK] [MASK]an, [MASK] a body length up to 60 centimetres ( 24 in ) and weighing 
       pos   2: True=[##mar       ]  Predict=[##mal       ]  false  top3=[##mal / ##rt / ##mar]
       pos   9: True=[crust       ]  Predict=[freshwater  ]  false  top3=[freshwater / marine / small]
       pos  10: True=[##ace       ]  Predict=[##zo        ]  false  top3=[##zo / ro / ko]
       pos  13: True=[with        ]  Predict=[

In [ ]:
import torch
from transformers import BertForMaskedLM
from utils import (replace_layers, eval_mlm, MLMDataset,
                   save_result, print_result,
                   load_base_params, make_result_row)

EXP_ID        = "T10"
DESCRIPTION   = "FFN, ALL layers"
REPLACE_Q     = False
REPLACE_K     = False
REPLACE_V     = False
REPLACE_FFN   = True
LAYER_IDS     = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
DISTILL_STEPS = 3000
N_SAMPLES     = 300
BATCH_SIZE    = 16
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def main():
    print(f"  {EXP_ID} - {DESCRIPTION}")
    print(f"  Device: {DEVICE}")

    base = load_base_params()
    print(f"  Baseline params   : {base['base_params']:,}")
    print(f"  Baseline accuracy : {base['baseline_accuracy']:.3f}%")
    print(f"  Baseline loss     : {base['baseline_loss']:.4f}")

    model = BertForMaskedLM.from_pretrained("bert-base-uncased")
    model.eval()
    replace_layers(
        model,
        layer_ids=LAYER_IDS,
        replace_q=REPLACE_Q,
        replace_k=REPLACE_K,
        replace_v=REPLACE_V,
        replace_ffn=REPLACE_FFN,
        distill_steps=DISTILL_STEPS,
        log_every=200,
    )


    dataset = MLMDataset(n_samples=N_SAMPLES)
    metrics = eval_mlm(model, dataset, batch_size=BATCH_SIZE, device=DEVICE)

    result = make_result_row(
        exp_id=EXP_ID, desc=DESCRIPTION,
        replace_q=REPLACE_Q, replace_k=REPLACE_K,
        replace_v=REPLACE_V, replace_ffn=REPLACE_FFN,
        layers=LAYER_IDS, metrics=metrics,
        base_params=base["base_params"],
    )

    print_result(result)
    save_result(result)
    print(f"\n  {EXP_ID} done")


if __name__ == "__main__":
    main()

  T10 - FFN, ALL layers
  Device: cuda
  Baseline params   : 109,514,298
  Baseline accuracy : 63.377%
  Baseline loss     : 2.2498


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Layer 0  ----------------------------------------

  Distilling L0.FFN-inter  [768 -> 3072]  (2,362,368 params)
    Step      TotalLoss    MSE          Gate         EyeMSE       Viol%    LR
        0    8.4790e-01    5.5934e-01    2.0013e-01    7.2790e-03  100.0%  2.0e-03
      200    4.5770e-01    3.3876e-01    1.0748e-01    2.2543e-03   68.6%  2.0e-03
      400    4.5271e-01    3.3414e-01    1.0742e-01    2.2466e-03   68.8%  1.9e-03
      600    4.4552e-01    3.2821e-01    1.0617e-01    2.2336e-03   68.2%  1.8e-03
      800    4.5063e-01    3.3309e-01    1.0633e-01    2.2369e-03   68.2%  1.7e-03
     1000    4.4710e-01    3.2982e-01    1.0602e-01    2.2107e-03   67.9%  1.5e-03
     1200    4.4940e-01    3.3182e-01    1.0659e-01    2.2537e-03   68.4%  1.3e-03
     1400    4.4757e-01    3.2991e-01    1.0637e-01    2.3008e-03   68.1%  1.1e-03
     1600    4.4696e-01    3.2991e-01    1.0568e-01    2.3059e-03   67.8%  8.9e-04
     1800    4.4463e-01    3.2765e-01    1.0546e-01    2.200